In [9]:
import numpy as np

def line_gauss_rule_info(p,i):
    """
    INPUT: pth rule, ith point 
    PROCESS: calculates the ith point of the Legendre-Gauss quadrature for a pth rule (up to the 4th rule)
    OUTPUT: point, weight
    """
    if p == 1:
        table = [(0.0 , 2.0)]

    elif p == 2:
        table = [(-1/(np.sqrt(3)), 1.0),
                 (1/(np.sqrt(3)), 1.0)]

    elif p == 3:
        table = [(-np.sqrt(3.0/5.0), 5.0/9.0), 
                 (0.0, 8.0/9.0), 
                 (np.sqrt(3.0/5.0), 5.0/9.0)]

    elif p == 4:
        xi3 = np.sqrt((3-2*np.sqrt(6/5))/7)
        xi4 = np.sqrt((3+2*np.sqrt(6/5))/7)
        w1 = 0.5 - (1/6)*np.sqrt(5/6)
        w2 = 0.5 + (1/6)*np.sqrt(5/6)
        table = [(-xi4, w1), 
                 (-xi3, w2), 
                 (xi3, w2), 
                 (xi4, w1)]

    return table[i-1]


def quad_gauss_rule_info(p, k):
    """
    INPUT: pth rule, k (visiting counter)

    At first, the idea of the visiting counter wasn't intuitive to me, here's a quick example table for a p=2:

    | k | i | j | 
    | - | - | - | 
    | 1 | 1 | 1 | 
    | 2 | 1 | 2 | 
    | 3 | 2 | 1 |
    | 4 | 2 | 2 | 

    
    PROCESS: calculates Legendre-Gauss in 2x2 
    OUTPUT: (xi, eta), weight (multiplied)
    """

    i = (k-1)//p + 1
    j = (k-1)%p + 1

    xi, w1 = line_gauss_rule_info(p, i)
    eta, w2 = line_gauss_rule_info(p, j)

    return (xi, eta) , w1*w2

def quad4_isoP_shape_fun_der(ncoor, qcoor):
    """
    INPUT: ncoor (node coordinates), qcoor (parametric coordinates)
    PROCESS:
    OUTPUT: Nf (shape functions), dNx, dNy (derivatives of the shape functions), Jdet (determinant of the jacobian)
    """
    xi, eta = qcoor
    
    Nf = 0.25 * np.array([
        (1-xi)*(1-eta),
        (1+xi)*(1-eta),
        (1+xi)*(1+eta),
        (1-xi)*(1+eta)
    ])

    dNxi = 0.25 * np.array([
        -(1-eta),
        (1-eta),
        (1+eta),
        -(1+eta)
    ])

    dNeta = 0.25 * np.array([
        -(1-xi),
        -(1+xi),
        (1+xi),
        (1-xi)
    ])

    ncoor = np.asarray(ncoor)

    x = ncoor[:, 0]
    y = ncoor[:, 1]

    J11 = np.dot(dNxi, x)
    J12 = np.dot(dNxi, y)
    J21 = np.dot(dNeta, x)
    J22 = np.dot(dNeta, y)

    Jdet = J11*J22 - J12*J21

    # to get ∂N/∂x, remember that chain rule gives ∂N/∂x= ∂N/∂ξ * ∂ξ/∂x + ∂N/∂η * ∂η/∂x, in matrix form, its thus the Jacobian inverse.

    dNx = ( J22 * dNxi - J12 * dNeta) / Jdet
    dNy = (-J21 * dNxi + J11 * dNeta) / Jdet

    return Nf,  dNx, dNy, Jdet

def quad4_isoP_membrane_stiffness(ncoor, mprop, fprop, options):
    """
    INPUT: ncoor (node coordinates), 
           mprop (material properties, given as list {Emat, rho, alpha}, note that Emat is 3x3 matrix), also rho and alpha are not used here)
           fprop (fabrication properties, i.e, thickness as four entry list, one entry list or empty list). If empty, it assumes uniform h = 1
           options (processing options, can have two items. numer (true or false) not used in this python implementation, p specifies Gauss rule.
    PROCESS:
    OUTPUT:
    """
    Emat = np.asarray(mprop[0], dtype=float)

    if len(options) == 2:
        _, p = options
    else:
        p = 2

    if len(fprop) == 0:
        thickness = [1.0]
    else:
        thickness = fprop[0] if isinstance(fprop[0], (list, tuple, np.ndarray)) else [fprop[0]] 


    Ke = np.zeros((8, 8))

    for k in range(1, p*p + 1):
        
        qcoor, w = quad_gauss_rule_info(p, k) # obtain isoparametric coordinates in the points where Gauss integration will be performed
        
        Nf, dNx, dNy, Jdet = quad4_isoP_shape_fun_der(ncoor, qcoor)

        c = w* Jdet * thickness[0]

        #displacement strain matrix
        B = np.zeros((3,8))

        for i in range(4): # Felippa uses Flatten in Mathematica after creating tables with pairs (dNx, 0), which removes the braces. Here it's just a for loop
            B[0, 2*i]     = dNx[i]
            B[1, 2*i + 1] = dNy[i]
            B[2, 2*i]     = dNy[i]
            B[2, 2*i + 1] = dNx[i]

        Ke += c * (B.T @ Emat @ B)

    return Ke
        

In [12]:
# Exercise 17.1

Em = 48
a = 4
b = 2
h = 1
nu = 0 #poisson

ncoor = [(0,0),
         (a,0),
         (a,b),
         (0,b)]

Emat = [[Em/(1 - nu**2),    nu*Em/(1 - nu**2), 0],
        [nu*Em/(1 - nu**2), Em/(1 - nu**2),    0],
        [0, 0, Em*(1 - nu)/(2*(1 - nu**2))]]


for p in [1, 2, 3, 4]:
    Ke = quad4_isoP_membrane_stiffness(ncoor, [Emat, 0, 0], [h], (True, p))

    print(f"Gauss integration rule: {p} x {p}")
    
    Ke_chop = np.where(np.abs(Ke) < 1e-10, 0.0, Ke)

    print("Ke =")
    print(Ke_chop)

    eigvals = np.linalg.eigvalsh(Ke)
    eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)

    print("Eigenvalues of Ke =")
    print(eigvals)



Gauss integration rule: 1 x 1
Ke =
[[ 18.   6.   6.  -6. -18.  -6.  -6.   6.]
 [  6.  27.   6.  21.  -6. -27.  -6. -21.]
 [  6.   6.  18.  -6.  -6.  -6. -18.   6.]
 [ -6.  21.  -6.  27.   6. -21.   6. -27.]
 [-18.  -6.  -6.   6.  18.   6.   6.  -6.]
 [ -6. -27.  -6. -21.   6.  27.   6.  21.]
 [ -6.  -6. -18.   6.   6.   6.  18.  -6.]
 [  6. -21.   6. -27.  -6.  21.  -6.  27.]]
Eigenvalues of Ke =
[ 0.  0.  0.  0.  0. 24. 60. 96.]
Gauss integration rule: 2 x 2
Ke =
[[ 24.   6.   0.  -6. -12.  -6. -12.   6.]
 [  6.  36.   6.  12.  -6. -18.  -6. -30.]
 [  0.   6.  24.  -6. -12.  -6. -12.   6.]
 [ -6.  12.  -6.  36.   6. -30.   6. -18.]
 [-12.  -6. -12.   6.  24.   6.   0.  -6.]
 [ -6. -18.  -6. -30.   6.  36.   6.  12.]
 [-12.  -6. -12.   6.   0.   6.  24.  -6.]
 [  6. -30.   6. -18.  -6.  12.  -6.  36.]]
Eigenvalues of Ke =
[ 0.  0.  0. 24. 24. 36. 60. 96.]
Gauss integration rule: 3 x 3
Ke =
[[ 24.   6.   0.  -6. -12.  -6. -12.   6.]
 [  6.  36.   6.  12.  -6. -18.  -6. -30.]
 [  0.   6.